# Trying o classify on doppler bf output
- https://medium.com/data-bistrot/a-simple-image-classifier-with-a-python-neural-network-82a5522fe48b

In [21]:
import torch as tch
import torchvision as tv
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [47]:
batch_size = 3 # num images
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(10),
    #transforms.RandomAffine(translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), # adjust rgb
])

In [48]:
train_dataset = tv.datasets.ImageFolder(
    "class_images/train",
    transform=transform
)

val_dataset = tv.datasets.ImageFolder(
    "class_images/val",
    transform=transform
)

train_loader = tch.utils.data.DataLoader(train_dataset, batch_size, shuffle=True)
val_loader = tch.utils.data.DataLoader(val_dataset, batch_size)

In [49]:
classes = ('jerry', 'no_jerry')

In [52]:
class RatCNN(tch.nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = tch.nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = tch.nn.Conv2d(16, 32, 3, padding=1)

        self.pool = tch.nn.MaxPool2d(2, 2)

        self.fc1 = tch.nn.Linear(32 * 16 * 16, 64)
        self.fc2 = tch.nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool(tch.nn.functional.relu(self.conv1(x)))
        x = self.pool(tch.nn.functional.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = tch.nn.functional.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [53]:
model = RatCNN()

criterion = tch.nn.CrossEntropyLoss()
optimizer = tch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):

    model.train()

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    print(f"Epoch {epoch} complete")

Epoch 0 complete
Epoch 1 complete
Epoch 2 complete
Epoch 3 complete
Epoch 4 complete
Epoch 5 complete
Epoch 6 complete
Epoch 7 complete
Epoch 8 complete
Epoch 9 complete


In [54]:
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", correct / total)

Accuracy: 0.5


In [55]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(all_labels, all_preds))

[[2 0]
 [2 0]]
